# Chapter 16 – The Master Theorem: calls and recurrences against the solution (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch16_master_theorem_cpp.ipynb)

The chapter contains no algorithm with pseudo-code: this notebook is therefore short and **measures** what the Master Theorem predicts. First we count the calls of `sumRecHalving`, the halving sum algorithm of the course code, whose recurrence $T(n)=2T(n/2)+1$ falls in case 1 ($\Theta(n)$, in fact exactly $2n-1$ calls); then we compute directly some recurrences $T(n)=aT(n/b)+f(n)$ of the examples of the chapter and check that the ratio with the function given by the Master Theorem tends to a constant.

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/recursion/16-master-theorem/).

Each program is written to a file by the `%%writefile` cell and then compiled and run with `g++ -std=c++17 -O2`: run the cells in order, from top to bottom. No external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `sumRecHalving` → `sumRecHalving`.

## 1. The calls of `sumRecHalving` (case 1)

`sumRecHalving`$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the sum of `a[l..r]`, obtained by adding the sums of the two halves. Every call does a constant amount of work, so $T(n)=2T(n/2)+1$: $a=2$, $b=2$, $n^{\log_b a}=n$, case 1, $T(n)=\Theta(n)$. The chapter reports $15$, $1\,999$ and $2\,097\,151$ calls for $n=8$, $1000$ and $2^{20}$, that is exactly $2n-1$.

The array contains $1,2,\dots,n$, so the expected sum is $n(n+1)/2$.

In [1]:
%%writefile sum_rec_halving.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int calls = 0;   // number of calls

// Sum of a[l..r] (0-based indices) by halving
long long sumRecHalving(int *a, int l, int r)
{
    calls++;
    if (l == r)
    {
        return a[l];
    }
    int mid = l + (r - l) / 2;
    return sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
}

#ifndef NOMAIN
int main()
{
    int ns[] = {8, 1000, 1 << 20};
    for (int n : ns)
    {
        vector<int> a(n);
        for (int i = 0; i < n; i++)
        {
            a[i] = i + 1;
        }
        calls = 0;
        long long s = sumRecHalving(a.data(), 0, n - 1);
        printf("n=%-8d sum=%-14lld calls=%-9d 2n-1=%d\n", n, s, calls, 2 * n - 1);
    }
    return 0;
}
#endif

Writing sum_rec_halving.cpp


In [2]:
!g++ -std=c++17 -O2 -o sum_rec_halving sum_rec_halving.cpp && ./sum_rec_halving

n=8        sum=36             calls=15        2n-1=15
n=1000     sum=500500         calls=1999      2n-1=1999
n=1048576  sum=549756338176   calls=2097151   2n-1=2097151


**Try it yourself.** Change `n` (also to a value that is not a power of 2) and check that the calls are always $2n-1$.

In [3]:
%%writefile prova_sum_rec_halving.cpp
#define NOMAIN
#include "sum_rec_halving.cpp"

int main()
{
    // Try an n that is not a power of 2: the calls are still 2n-1.
    int n = 777;
    vector<int> a(n);
    for (int i = 0; i < n; i++) a[i] = i + 1;
    calls = 0;
    long long s = sumRecHalving(a.data(), 0, n - 1);
    printf("%lld %d %d\n", s, calls, 2 * n - 1);
    return 0;
}

Writing prova_sum_rec_halving.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_sum_rec_halving prova_sum_rec_halving.cpp && ./prova_sum_rec_halving

302253 1553 1553


## 2. Recurrences of the examples of the chapter and the Master Theorem

For $T(n)=a\,T(n/b)+f(n)$, $T(1)=1$ and $n=b^k$, the function `valuta` computes $T(n)$ from the definition ($k$ steps: $T(b^{i})=a\,T(b^{i-1})+f(b^{i})$). The Master Theorem gives the order of growth $\Theta(g)$ with $f(n)=n^p$ and $n^{\log_b a}=a^k$:

- **case 1** ($f$ grows slower): $g=n^{\log_b a}=a^k$;
- **case 2** ($f$ and $n^{\log_b a}$ of the same order): $g=f(n)\cdot\log_b n=n^p\,k$;
- **case 3** ($f$ grows faster): $g=f(n)=n^p$.

The last column is $T(n)/g(n)$: if the Master Theorem is right, it is a bounded number away from 0 (for large $n$ it gets close to a constant).

In [5]:
%%writefile master_theorem.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long IPow(long long x, int e)
{
    long long r = 1;
    for (int i = 0; i < e; i++)
    {
        r *= x;
    }
    return r;
}

// T(n) = a*T(n/b) + n^p, T(1) = 1, for n = b^k
void Valuta(int a, int b, int p, int k, long long &n, long long &t)
{
    n = 1;
    t = 1;
    for (int i = 0; i < k; i++)
    {
        n *= b;
        t = a * t + IPow(n, p);
    }
}

struct Riga
{
    const char *nome;
    int a, b, p, k, caso;
    const char *tipo;
    const char *gname;
};

#ifndef NOMAIN
int main()
{
    // (recurrence, a, b, p, k, case, kind of g, name of g)
    vector<Riga> righe = {
        {"T(n)=T(n/2)+1",      1, 2, 0, 20, 2, "level",  "log n"},
        {"T(n)=2T(n/2)+1",     2, 2, 0, 20, 1, "leaves", "n"},
        {"T(n)=2T(n/2)+n",     2, 2, 1, 20, 2, "level",  "n log n"},
        {"T(n)=3T(n/2)+n",     3, 2, 1, 20, 1, "leaves", "n^log2(3)"},
        {"T(n)=7T(n/2)+n^2",   7, 2, 2, 20, 1, "leaves", "n^log2(7)"},
        {"T(n)=16T(n/4)+n",   16, 4, 1, 10, 1, "leaves", "n^2"},
        {"T(n)=8T(n/2)+n^3",   8, 2, 3, 15, 2, "level",  "n^3 log n"},
        {"T(n)=8T(n/3)+n^2",   8, 3, 2, 12, 3, "f",      "n^2"},
    };
    printf("%-18s case %8s %20s  %-10s T/g\n", "recurrence", "n", "T(n)", "Theta(g)");
    for (const Riga &r : righe)
    {
        long long n, t;
        Valuta(r.a, r.b, r.p, r.k, n, t);
        string tipo = r.tipo;
        long long g = tipo == "leaves" ? IPow(r.a, r.k) : (tipo == "level" ? IPow(n, r.p) * r.k : IPow(n, r.p));
        printf("%-18s %4d %8lld %20lld  %-10s %.3f\n", r.nome, r.caso, n, t, r.gname, (double)t / (double)g);
    }
    return 0;
}
#endif

Writing master_theorem.cpp


In [6]:
!g++ -std=c++17 -O2 -o master_theorem master_theorem.cpp && ./master_theorem

recurrence         case        n                 T(n)  Theta(g)   T/g
T(n)=T(n/2)+1         2  1048576                   21  log n      1.050
T(n)=2T(n/2)+1        1  1048576              2097151  n          2.000
T(n)=2T(n/2)+n        2  1048576             22020096  n log n    1.050
T(n)=3T(n/2)+n        1  1048576          10458256051  n^log2(3)  2.999
T(n)=7T(n/2)+n^2      1  1048576   186180488678924301  n^log2(7)  2.333
T(n)=16T(n/4)+n       1  1048576        1466015154176  n^2        1.333
T(n)=8T(n/2)+n^3      2    32768      562949953421312  n^3 log n  1.067
T(n)=8T(n/3)+n^2      3   531441        1992110014441  n^2        7.053


**Try it yourself.** Change the parameters $a$, $b$, $p$, $k$ and compare $T(n)$ with the function of the Master Theorem case.

In [7]:
%%writefile prova_master_theorem.cpp
#define NOMAIN
#include "master_theorem.cpp"

int main()
{
    // Try a recurrence of your own: T(n) = 4T(n/2) + n (case 1, g = n^2) with k = 10.
    long long n, t;
    Valuta(4, 2, 1, 10, n, t);
    printf("%lld %lld %.3f\n", n, t, (double)t / (double)IPow(4, 10));
    return 0;
}

Writing prova_master_theorem.cpp


In [8]:
!g++ -std=c++17 -O2 -o prova_master_theorem prova_master_theorem.cpp && ./prova_master_theorem

1024 2096128 1.999
